<a href="https://colab.research.google.com/github/Samarbal/Big-Data-Analysis/blob/main/Big_Data_Lab_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## LAB 2 EXERCISE

In [ ]:
# import labraries
import pandas as pd
import numpy as np
import time
import os

### File Format Comparison

based on :

 1. Write time: How long it takes to save the DataFrame.
 2. Read time: How long it takes to load the DataFrame.
 3. File size: The size of the saved file.
 4. Query performance:  Time taken for a filter and aggregation operation



In [ ]:
# data size
n_rows = 2000000

In [ ]:
files_data = pd.DataFrame({
 'id': range(n_rows),
 'timestamp': pd.date_range('2023-01-01', periods=n_rows, freq='1s'),
 'category': np.random.choice(['A', 'B', 'C', 'D', 'E'], n_rows),
 'value1': np.random.randn(n_rows),
 'value2': np.random.randn(n_rows),
 'value3': np.random.uniform(0, 100, n_rows),
 'text': [f'Transaction_{i}' for i in range(n_rows)]
})

In [ ]:
# size of the data in memory
mem_size = files_data.memory_usage().sum() / 1024**2
print(f"Memory size in RAM {mem_size:.2f} MB")

Memory size in RAM 106.81 MB


In [ ]:
file_formats = ['csv', 'json', 'parquet', 'feather', 'pickle']
results = []

In [ ]:
# Comparison
for format in file_formats:
    filename = f'temp_data.{format}'

    # measure write time
    start_time = time.time()
    # if csv format
    if format == 'csv':
        files_data.to_csv(filename, index=False)
     # if json format
    elif format == 'json':
        files_data.to_json(filename, orient='records', lines=True)
     # if parquet format
    elif format == 'parquet':
      files_data.to_parquet(filename, index=False)
     # if feather format
    elif format == 'feather':
      files_data.to_feather(filename)
    # if pickle format
    elif format == 'pickle':
      files_data.to_pickle(filename)
      # write time
    write_time = time.time() - start_time

    # ----------------------------------------
    # measure file size

    file_size_mb = os.path.getsize(filename)/ (1024 * 1024)

    # ----------------------------------
    # compression ratio
    compression_ratio = mem_size / file_size_mb if file_size_mb > 0 else 0

    #-------------------------------
    # measure read time
    start_time = time.time()
    if format == 'csv':
        df_read = pd.read_csv(filename)
    elif format == 'json':
        df_read = pd.read_json(filename, orient='records', lines=True)
    elif format == 'parquet':
        df_read = pd.read_parquet(filename)
    elif format == 'feather':
        df_read = pd.read_feather(filename)
    elif format == 'pickle':
        df_read = pd.read_pickle(filename)
    read_time = time.time() - start_time


    #----------------------------------------------
    # Query performance (filter + aggregation)
    start_time = time.time()
    filter_aggregate=  df_read[df_read['category'] == 'A']['value1'].mean()
    query_time = time.time() - start_time

    #----------------------------------------
    # store results
    results.append({
        'Format': format.capitalize(),
        'Write Time (s)': write_time,
        'Size (MB)': file_size_mb,
        'Read Time (s)': read_time,
        'Query Time (s)': query_time,
        'Comparsion Ratio': compression_ratio })


comparison_df = pd.DataFrame(results)

NameError: name 'os' is not defined

In [ ]:
print(f'Orignal data size : {mem_size}')
print(".................")
display(comparison_df)

Orignal data size : 106.81164932250977
.................


,Format,Write Time (s),Size (MB),Read Time (s),Query Time (s)
0,Csv,23.829497,202.821705,3.862952,0.217710
1,Json,9.236093,288.878857,23.894988,0.401089
2,Parquet,2.299047,78.361027,2.644763,0.218634
3,Feather,0.626327,90.112749,1.098503,0.207812
4,Pickle,0.874971,121.025637,0.437739,0.206621


#### Analysis after the comparasion

1. **Storage Efficiency**: Parquet is the most efficient format, significantly reducing the file size to 78.36 MB. This highlights the advantage of Columnar Storage in optimizing storage costs.

2. **I/O Performance**: Feather achieved the fastest write time (0.62s), making it ideal for temporary data storage. Meanwhile, Pickle excelled in read speed (0.43s), ensuring rapid memory loading.

3. **Binary vs. Textual**: Traditional formats like CSV and JSON showed poor performance, with higher latency and larger file sizes, proving they are less suitable for high-scale Big Data Pipelines.

4. **Query Stability**: The consistent Query Time across all formats suggests that the primary performance bottleneck lies in Disk I/O operations rather than data retrieval itself.




### File Format Comparison Metrics

This section will compare various file formats (CSV, JSON, Parquet, Feather, Pickle) based on:
1.  **Write time**: How long it takes to save the DataFrame.
2.  **Read time**: How long it takes to load the DataFrame.
3.  **File size**: The size of the saved file.
4.  **Query performance**: Time taken for a filter and aggregation operation (`SELECT AVG(value1) FROM table_name WHERE category = 'A'`).
5.  **Compression Ratio**: How much the file is compressed compared to the original data in memory.